# Web Scraping of RI UFRN

## 1. Importing the required libraries

In [ ]:
# Importing the required libraries.
import re, traceback, csv, pandas as pd, time, requests, numpy as np
import playwright._impl._errors as errors
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from playwright.async_api import async_playwright
from twisted.internet.error import TCPTimedOutError, TimeoutError

## 2. Defining the Spider class

In [ ]:
class Spider_RI_UFRN:
    def __init__(self, url):
        self.__url_base = url
        self.__url_api = "https://repositorio.ufrn.br/server/api/core/items/"
        self.__max_attempts = 1
        self.__user_agent = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) "
                             "Chrome/122.0.0.0 Safari/537.36 OPR/108.0.0.0")
        self.__data = None
        self.__playwright = None
        self.__browser = None
        self.__page = None

    @property
    def get_data(self):
        return self.__data

    async def __get_html(self, url, css_selector=None, to_close=True):
        if self.__playwright is None:
            self.__playwright = await async_playwright().start()
        if self.__browser is None or not self.__browser.is_connected():
            self.__browser = await self.__playwright.chromium.launch(headless=True, args=["--start-maximized"])
        if self.__page is None or self.__page.is_closed():
            self.__page = await self.__browser.new_page(user_agent=self.__user_agent)
            # self.__page = await self.__browser.new_page()
        await self.__page.goto(url)
        if css_selector is not None:
            await self.__page.wait_for_selector(css_selector)
        html = await self.__page.content()
        if to_close:
            await self.__browser.close()
            await self.__playwright.stop()
        return html

    async def __parse_categories(self, num_attempt=0):
        try:
            # Initializing the webdriver.
            await self.__get_html(self.__url_base, to_close=False)

            # Enabling the cookies.
            css = "div#klaro > div > div > div > div > div > button.cm-btn.cm-btn-success"
            button = self.__page.locator(css)
            await button.wait_for(state="visible", timeout=120000)
            if await button.is_visible() and await button.is_enabled():
                await button.click()

            # Getting the relative URLs of the categories (PhD and MSc' Thesis).
            css = "a.lead.ng-star-inserted"
            links = self.__page.locator(css)
            links = await links.all()
            self.__data = [urljoin(self.__page.url, await i.get_attribute("href"))
                           for i in links]
        except (errors.TimeoutError, errors.TargetClosedError, errors.Error, AttributeError, Exception,
                TCPTimedOutError, TimeoutError) as e:
            print(f"[ERROR-DEBUG] {e}: {self.__url_base}")
            print("".join(traceback.format_tb(e.__traceback__)))
            if num_attempt <= self.__max_attempts:
                num_attempt += 1
                print(f"Number of attempting in 'parse_categories': {num_attempt}")
                await self.parse_categories(num_attempt)

    async def __parse_thesis_links(self, num_attempt=0):
        try:
            links = list()
            for link in self.__data:
                # Redirecting the list of works.
                await self.__page.goto(link)

                # Getting the relative URLs of the documents.
                css = "a[aria-label='Next']"
                button = self.__page.locator(css)
                await button.wait_for(state="visible", timeout=120000)
                flag = True
                while flag:
                    # Scrolling down the mouse until the page's end.
                    for _ in range(20):
                        await self.__page.mouse.wheel(0, 300)
                        time.sleep(0.5)

                    # Colleting the links of a list page.
                    soup = BeautifulSoup(await self.__page.content(), "html.parser")
                    urls = soup.select("div#p-cp > ul > li[data-test='list-object']")
                    urls = [urljoin(self.__page.url, i.find("a")["href"])
                            for i in urls if i.find("a").has_attr("href")]
                    if len(urls) > 0:
                        links.extend(urls)

                    # Navigating among the next pages.
                    flag = await button.is_visible() and await button.is_enabled()
                    if flag:
                        await button.click()
                    else:
                        flag = False
                    time.sleep(5)

            # Closing the webdriver.
            await self.__browser.close()
            await self.__playwright.stop()

            self.__data = links
        except (errors.TimeoutError, errors.TargetClosedError, errors.Error, AttributeError, Exception,
                TCPTimedOutError, TimeoutError) as e:
            print(f"[ERROR-DEBUG] {e}: {self.__url_base}")
            print("".join(traceback.format_tb(e.__traceback__)))
            if num_attempt <= self.__max_attempts:
                num_attempt += 1
                print(f"Number of attempting in 'parse_thesis_links': {num_attempt}")
                await self.parse_thesis_links(num_attempt)

    async def __parse_items(self, num_attempt=0):
        try:
            for idx, link in enumerate(self.__data):
                record = dict()
                # Requesting the metadata.
                link = f"{self.__url_api}{link.split("/")[-1]}"
                result = requests.get(link)
                result = result.json()

                # Title.
                record["title"] = None if "name" not in result \
                    else re.sub(r"\s+", " ", result["name"]).strip()

                # Advisee.
                record["advisee"] = None if "dc.contributor.author" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for i in result["metadata"]["dc.contributor.author"]])

                # Advisor.
                record["main_advisor"] = None if "dc.contributor.advisor" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for i in result["metadata"]["dc.contributor.advisor"]])

                # Co-Advisor.
                record["co_advisor"] = None if "dc.contributor.advisor-co1" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for c in [f"dc.contributor.advisor-co{n}" for n in range(1, 6)] \
                                    if c in result["metadata"]
                                for i in result["metadata"][c]])

                # Committee or Referees.
                record["referees"] = None if "dc.contributor.referees1" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for c in [f"dc.contributor.referees{n}" for n in range(1, 11)] \
                                    if c in result["metadata"]
                                for i in result["metadata"][c]])

                # Defense Date.
                record["defense_date"] = None if "dc.date.issued" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for i in result["metadata"]["dc.date.issued"]])

                # Affiliation.
                record["affiliation"] = None if "dc.publisher" not in result["metadata"] \
                                            else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                                for i in result["metadata"]["dc.publisher"]])

                # PT-BR Abstract.
                record["ptbr_abstract"] = None if "dc.description.resumo" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for i in result["metadata"]["dc.description.resumo"]])

                # English Abstract.
                record["en_abstract"] = None if "dc.description.abstract" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for i in result["metadata"]["dc.description.abstract"]])

                # Sponsorship.
                record["sponsorship"] = None if "dc.description.sponsorship" not in result["metadata"] \
                                            else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                               for i in result["metadata"]["dc.description.sponsorship"]])

                # Citation.
                record["citation"] = None if "dc.identifier.citation" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for i in result["metadata"]["dc.identifier.citation"]])

                # Document URI.
                record["document_uri"] = None if "dc.identifier.uri" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for i in result["metadata"]["dc.identifier.uri"]])

                # Program.
                record["program"] = None if "dc.publisher.program" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for i in result["metadata"]["dc.publisher.program"]])

                # Keywords.
                record["auth_keywords"] = None if "dc.subject" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for i in result["metadata"]["dc.subject"]])

                # Work Type.
                record["work_type"] = None if "dc.type" not in result["metadata"] \
                    else tuple([re.sub(r"\s+", " ", i["value"]).strip()
                                for i in result["metadata"]["dc.type"]])

                self.__data[idx] = record
        except (AttributeError, Exception, TCPTimedOutError, TimeoutError) as e:
            print(f"[ERROR-DEBUG] {e}: {self.__url_base}")
            print("".join(traceback.format_tb(e.__traceback__)))
            if num_attempt <= self.__max_attempts:
                num_attempt += 1
                print(f"Number of attempting in 'parse_items': {num_attempt}")
                await self.parse_items(num_attempt)

    async def collect_data(self):
        await self.__parse_categories()
        await self.__parse_thesis_links()
        await self.__parse_items()
        return self.get_data

## 3. Executing the Spider

In [ ]:
# Determining the URL of target page.
url = "https://repositorio.ufrn.br/handle/123456789/11949"

# Creating the spider.
spider = Spider_RI_UFRN(url)

# Crawling the data.
data = await spider.collect_data()

In [ ]:
# Printing the number of records collected.
print("Number of records collected: {}.".format(len(data)))

## 4. Preprocessing the data

In [ ]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text, flags=re.IGNORECASE).strip()
    text = text.replace("- ", "-").replace("\ufeff", "")
    return text

In [ ]:
# Creating the Pandas' DataFrame object.
df_data = pd.DataFrame(data)

In [ ]:
# Listing the five first records.
df_data.head()

In [ ]:
# Checking the information about the dataset.
df_data.info()

In [ ]:
# Handling the None values.
df_data.replace({np.nan: None}, inplace=True)

In [ ]:
# Normalizing the content of features.
df_data = df_data.apply(lambda row: row.apply(
    lambda x: x[-1] if x is not None and \
        len(x) == 1 else x))

In [ ]:
# Fixing the missing values for the "main_advisor" feature.
df_data.loc[df_data.document_uri == "https://repositorio.ufrn.br/handle/123456789/60905",
            "main_advisor"] = "Silva, Ivanovitch Medeiros Dantas da"
df_data.loc[df_data.document_uri == "https://repositorio.ufrn.br/handle/123456789/45590",
            "main_advisor"] = "Sousa Junior, Vicente Angelo de"

In [ ]:
# Fixing the missing values for the "affiliation" feature.
df_data.loc[df_data.affiliation.isnull(), "affiliation"] = "Universidade Federal do Rio Grande do Norte"

In [ ]:
# Fixing the missing values for the "referees" feature.
values = {"https://repositorio.ufrn.br/jspui/handle/123456789/15171": ("Calôba, Luis Pereira", "Tozzi, Clécio Luis",
                                                                       "Gonçalves, Luiz Marcos Garcia", "Melo, Jorge Dantas de"),
          "https://repositorio.ufrn.br/jspui/handle/123456789/15114": ("Gonçalves, Luiz Marcos Garcia",
                                                                       "Botelho, Silvia Silva da Costa",
                                                                       "Freira, Eduardo Oliveira", "Alsina, Pablo Javier"),
          "https://repositorio.ufrn.br/jspui/handle/123456789/18564": ("Paiva, José Álvaro de", "Martins, Allan de Medeiros",
                                                                       "Ribeiro, Ricardo Lúcio de Araújo"),
          "https://repositorio.ufrn.br/jspui/handle/123456789/15487": ("Oliveira, José Tavares de", "Silva Júnior, José Luiz da",
                                                                       "Pimentel Filho, Max Chianca",
                                                                       "Oliveira, Arrhenius Vinicius da Costa"),
          "https://repositorio.ufrn.br/jspui/handle/123456789/15284": ("Valentim, Ricardo Alexandro de Medeiros",
                                                                       "Guerreiro, Ana Maria Guimarães"),
          "https://repositorio.ufrn.br/jspui/handle/123456789/15262": ("Alves, Raimundo Nazareno Cunha",
                                                                       "Araújo, Aldayr Dantas de",
                                                                       "Maitelli, André Laurindo"),
          "https://repositorio.ufrn.br/jspui/handle/123456789/15172": ("Oliveira, Luiz Affonso Henderson Guedes de",
                                                                       "Brayner, Ângelo Roncalli Alencar",
                                                                       "Dória Neto, Adrião Duarte")}
df_data.loc[df_data.document_uri.isin(list(values.keys())), "referees"] = \
df_data.loc[df_data.document_uri.isin(list(values.keys())), "document_uri"].apply(
    lambda x: values[x]
)

In [ ]:
# Fixing the missing values for the "citation" feature.
df_data.loc[df_data.document_uri == "https://repositorio.ufrn.br/jspui/handle/123456789/15285", "citation"] = \
    ("VIEGAS, Carlos Manuel Dias. Análise de Desempenho de Estratégias de "
     "Retransmissão para o Mecanismo HCCA do Padrão de Redes Sem Fio 802.11e. "
     "2009. 96 f. Dissertação (Mestrado em Engenharia de Computação; Redes sem "
     "fio; Redes industriais) - Universidade Federal do Rio Grande do Norte, "
     "Natal, 2009.")

In [ ]:
# Normalizing the "work_type" feature.
df_data.work_type = df_data.work_type.apply(
    lambda x: "PhD" if "doctoral" in x else \
    "MSc" if "master" in x else "Other")

In [ ]:
# Normalizing the "advisee", "main_advisor", "co_advisor", and "referees" features.
cols = ["advisee", "main_advisor", "co_advisor", "referees"]
df_data.loc[:, cols] = df_data.loc[:, cols].apply(
    lambda row: row.apply(
        lambda x: f"{x.split(",")[-1]} {x.split(",")[0]}".strip() \
            if isinstance(x, str) and "," in x else \
            tuple([f"{i.split(",")[-1]} {i.split(",")[0]}".strip()
                   for i in x]) if isinstance(x, tuple) else x))

In [ ]:
# Normalizing the column "auth_keywords".
df_data.loc[df_data.auth_keywords.notnull(), "auth_keywords"] = df_data.loc[
    df_data.auth_keywords.notnull(), "auth_keywords"].apply(lambda x: tuple(
        [clean_text(k).strip() for k in x
            if len(clean_text(k).strip())]))

In [ ]:
# Normalizing the columns "title", "citation", "ptbr_abstract" and "en_abstract".
df_data.loc[df_data.title.notnull(), "title"] = df_data.loc[
    df_data.title.notnull(), "title"].apply(clean_text)
df_data.loc[df_data.citation.notnull(), "citation"] = df_data.loc[
    df_data.citation.notnull(), "citation"].apply(clean_text)
df_data.loc[df_data.ptbr_abstract.apply(lambda x: isinstance(x, tuple)),
            "ptbr_abstract"] = df_data.loc[
        df_data.ptbr_abstract.apply(lambda x: isinstance(x, tuple)),
            "ptbr_abstract"].apply(lambda x: x[-1])
df_data.loc[df_data.ptbr_abstract.notnull(), "ptbr_abstract"] = df_data.loc[
    df_data.ptbr_abstract.notnull(), "ptbr_abstract"].apply(clean_text)

df_data.loc[df_data.ptbr_abstract.notnull(), "ptbr_abstract"] = df_data.loc[
    df_data.ptbr_abstract.notnull(), "ptbr_abstract"].apply(clean_text)
df_data.loc[df_data.en_abstract.notnull(), "en_abstract"] = df_data.loc[
    df_data.en_abstract.notnull(), "en_abstract"].apply(clean_text)

In [ ]:
# Checking the information about the dataset.
df_data.info()

## 5. Saving the data

In [ ]:
# Saving the data into a CSV file.
df_data.to_csv("ppgeec_phd_msc_thesis.csv", header=0, index=False, quoting=csv.QUOTE_ALL)